# Agentic Memory Nav 学习笔记

这份 notebook 目标不是只看代码，而是帮助你顺着系统逻辑“从输入到输出”理解这个项目。

我们会学习 4 件事：

1. 这个项目的整体架构是什么，为什么是这种结构。
2. 每个核心模块如何协作：mapping、perception、scene graph、memory、planner、executor。
3. 代码中真正实现的“对象中心场景图”与你构想中的“点云中心场景图”有哪些差别。
4. 如何自己写代码做最小验证，确认这个实现是否真的成立。

> 这不是一个纯理论笔记，而是一个“边学边验证”的实战教程。

## 1. 先看一句最重要的话：这个项目当前实现的核心不是“节点=点云”

你提出的想法很有价值：

- 让 scene graph 的 node 直接承载 3D 点云
- 用点云作为几何原始表示
- 再用它支撑长期 memory 和 agent reasoning

但当前这个仓库并没有这么做。它现在的真实架构更像：

- `mapping` 负责生成点云
- `perception` 负责检测对象
- `scene_graph` 负责把对象组织成 graph
- `memory` 负责保存事件和语义事实
- `planner` 负责在 graph + memory 上做导航决策
- `executor` 负责安全地执行动作

也就是说：

- 点云是底层几何材料；
- graph node 是抽象对象/房间/区域；
- memory 不是直接存点云，而是存结构化记录和摘要。

这个差别非常关键，因为它决定了系统的工程复杂度、查询效率和可验证性。

In [ ]:
# 这一节：导入项目并确认能否正常运行
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

print("project_root:", project_root)
print("PYTHONPATH ready:", str(project_root / "src"))

try:
    import agentic_memory_nav
    print("import agentic_memory_nav: OK")
            "source": [
                "## 6. 最后一句话：当前代码是否实现了你的想法？"
            ]
except Exception as exc:
    print("import agentic_memory_nav failed:", type(exc).__name__, exc)

In [ ]:
# 这一节：验证项目的最小构造对象可以创建
from agentic_memory_nav.common.types import (
    FrameObservation,
    Pose3D,
        {
            "cell_type": "markdown",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "## 7. RGB-only Agentic Memory：从视频到稳定局部地图",
                "",
                "现在系统的运行时目标是只依赖前端 RGB 视频：",
                "",
                "```text",
                "RGB video -> LingBot depth + c2w + intrinsics -> backprojection",
                "         -> fixed-frame local submap -> overlap stability gate",
                "         -> spatial memory -> VLM objects / Pi -> scene graph",
                "         -> knowledge memory -> native reasoning",
                "```",
                "",
                "这里的关键不是要求相机不动。机器人前进时，每帧看到的点云质心必然变化；因此稳定性不能用“质心漂移小”来判断。",
                "",
                "当前实现使用相邻局部点云的对称最近邻残差：",
                "",
                "$$",
                "r(P_t, P_{t+1}) = \\frac{1}{2}(d(P_t, P_{t+1}) + d(P_{t+1}, P_t))",
                "$$",
                "",
                "其中 $d(A, B)$ 表示 $A$ 中每个点到 $B$ 最近点距离的平均值。一个窗口中最大的相邻残差不超过阈值，才会被提交为 spatial memory。",
                "",
                "LiDAR、GT depth 和 GT pose 只用于离线研究评测，不是这个 RGB-only agent 的运行时前提。"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# 构造三个连续的 depth+c2w 反投影结果，模拟 LingBot 的三个采样帧。",
                "from agentic_memory_nav.common.types import MappingUpdate, Pose3D",
                "from agentic_memory_nav.mapping.local_submap import LocalSubmapBuilder",
                "import numpy as np",
                "",
                "def make_mapping_update(frame_index, offset_x):",
                "    # 每帧都有相似的局部表面，只是机器人向 x 方向移动。",
                "    cloud = np.array([",
                "        [offset_x, 0.0, 1.0],",
                "        [offset_x + 0.02, 0.0, 1.0],",
                "        [offset_x, 0.02, 1.0],",
                "    ], dtype=np.float32)",
                "    return MappingUpdate(",
                "        frame_id=f'frame_{frame_index:04d}',",
                "        timestamp=float(frame_index),",
                "        camera_pose=Pose3D(position=(offset_x, 0.0, 0.0)),",
                "        depth=np.ones((2, 2), dtype=np.float32),",
                "        confidence=np.ones((2, 2), dtype=np.float32),",
                "        local_pointcloud=cloud,",
                "        global_pointcloud=cloud,",
                "        is_keyframe=True,",
                "        map_version=frame_index + 1,",
                "    )",
                "",
                "builder = LocalSubmapBuilder(",
                "    window_frames=3,",
                "    frame_stride=1,",
                "    stability_threshold_m=0.10,",
                ")",
                "",
                "for index, offset_x in enumerate((0.00, 0.03, 0.06)):",
                "    submap = builder.add(make_mapping_update(index, offset_x))",
                "",
                "print('submap committed:', submap is not None)",
                "print('stable:', submap.stable)",
                "print('frame ids:', submap.frame_ids)",
                "print('overlap residual (m):', round(submap.geometric_residual_m, 4))",
                "print('submap point count:', len(submap.points))",
                "assert submap.stable"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "上面的例子中机器人确实移动了，但相邻观察仍来自重叠表面，因此 local submap 可以稳定提交。",
                "",
                "下面故意制造一个错误几何帧。它可能来自 depth 突然失真、pose 失效或不连续的重定位。这个窗口不会被作为可信空间事实提交。"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "unstable_builder = LocalSubmapBuilder(",
                "    window_frames=3,",
                "    frame_stride=1,",
                "    stability_threshold_m=0.10,",
                ")",
                "",
                "unstable_builder.add(make_mapping_update(0, 0.00))",
                "unstable_builder.add(make_mapping_update(1, 0.03))",
                "unstable_submap = unstable_builder.add(make_mapping_update(2, 2.00))",
                "",
                "print('stable:', unstable_submap.stable)",
                "print('overlap residual (m):', round(unstable_submap.geometric_residual_m, 4))",
                "print('memory decision:', 'commit spatial memory' if unstable_submap.stable else 'keep as untrusted candidate')",
                "assert not unstable_submap.stable"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "## 8. 从 2D bbox 到物体点云 $P_i$",
                "",
                "局部子地图表示环境几何；物体点云 $P_i$ 表示某个语义对象的局部几何。",
                "",
                "当前最小可运行路径是：",
                "",
                "1. VLM 输出对象类别、属性和 2D bbox；",
                "2. bbox fallback 或未来的 SAM mask 产生二值 instance mask；",
                "3. mask 选择 depth 像素；",
                "4. depth + intrinsics + c2w 回投为世界坐标点；",
                "5. 压缩保存为 NPZ，并把引用写进 `ObjectObservation.geometry`。",
                "",
                "这里用本地确定性 bbox segmenter 演示数据契约。它不是最终分割模型，但完整展示了 $P_i$ 如何进入后续图和记忆。"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "from agentic_memory_nav.common.types import CameraIntrinsics, FrameObservation, ObjectObservation",
                "from agentic_memory_nav.geometry.pointcloud_store import PointCloudStore",
                "from agentic_memory_nav.perception.instance_segmentation import BoundingBoxSegmenter, InstanceGeometryEnricher",
                "from pathlib import Path",
                "",
                "rgb = np.zeros((48, 64, 3), dtype=np.uint8)",
                "depth = np.full((48, 64), 2.0, dtype=np.float32)",
                "frame = FrameObservation(",
                "    frame_id='pi_frame',",
                "    timestamp=0.0,",
                "    rgb=rgb,",
                "    depth=depth,",
                "    camera_intrinsics=CameraIntrinsics(60.0, 60.0, 32.0, 24.0, 64, 48),",
                "    camera_pose=Pose3D(),",
                ")",
                "mapping = make_mapping_update(0, 0.0)",
                "mapping.depth = depth",
                "mapping.confidence = np.ones_like(depth)",
                "",
                "observation = ObjectObservation(",
                "    observation_id='obs_red_cube',",
                "    category='cube',",
                "    attributes={'color': 'red'},",
                "    bbox_2d=(20, 12, 44, 36),",
                "    center_3d=(0.0, 0.0, 0.0),",
                "    dimensions_3d=(0.0, 0.0, 0.0),",
                "    confidence=0.9,",
                "    timestamp=0.0,",
                "    frame_id=frame.frame_id,",
                ")",
                "",
                "pi_store = PointCloudStore(Path('/tmp/agentic_memory_nav_notebook_pi'))",
                "enricher = InstanceGeometryEnricher(BoundingBoxSegmenter(), pi_store)",
                "enriched = enricher.enrich(frame, mapping, [observation])[0]",
                "",
                "print('Pi artifact:', enriched.geometry.artifact_path)",
                "print('Pi point count:', enriched.geometry.point_count)",
                "print('Pi centroid:', enriched.geometry.centroid_3d)",
                "print('Pi dimensions:', enriched.geometry.dimensions_3d)",
                "assert enriched.geometry is not None"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "## 9. $P_i$、Scene Graph、Knowledge Memory 与 Native Reasoner",
                "",
                "对象点云本身不会直接塞进 SQLite。系统保存的是 NPZ 工件引用、点数、中心、尺寸、置信度和来源。",
                "",
                "随后：",
                "",
                "- `SceneGraphUpdater` 将对象观测变为 graph node；",
                "- 几何规则产生 `inside`、`near` 等 relation evidence；",
                "- `KnowledgeMemory` 将 node 和三元组边投影为可检索事实；",
                "- `NativeReasoner` 根据实体标签、属性和 relation evidence 返回目标与证据。",
                "",
                "下面用一个 room 与带 $P_i$ 的 object 建立完整最小闭环。"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "from agentic_memory_nav.common.types import MemoryItem, MemoryType, new_id",
                "from agentic_memory_nav.memory.knowledge_memory import KnowledgeMemory",
                "from agentic_memory_nav.memory.sqlite_store import SQLiteMemory",
                "from agentic_memory_nav.reasoning.native_reasoner import NativeReasoner",
                "from agentic_memory_nav.scene_graph.graph import SceneGraph",
                "from agentic_memory_nav.scene_graph.updater import SceneGraphUpdater",
                "",
                "room_observation = ObjectObservation(",
                "    observation_id='obs_room',",
                "    category='kitchen',",
                "    attributes={'kind': 'room'},",
                "    bbox_2d=(0, 0, 64, 48),",
                "    center_3d=(0.0, 0.0, 2.0),",
                "    dimensions_3d=(6.0, 3.0, 6.0),",
                "    confidence=0.99,",
                "    timestamp=0.0,",
                "    frame_id='pi_frame',",
                ")",
                "",
                "graph = SceneGraph()",
                "updater = SceneGraphUpdater(graph)",
                "updater.update([room_observation, enriched])",
                "",
                "memory_path = Path('/tmp/agentic_memory_nav_notebook_knowledge.sqlite3')",
                "if memory_path.exists():",
                "    memory_path.unlink()",
                "memory = SQLiteMemory(memory_path)",
                "knowledge = KnowledgeMemory(memory)",
                "created = knowledge.materialize(graph)",
                "reasoner = NativeReasoner(knowledge)",
                "result = reasoner.resolve(graph, {'object': 'cube', 'color': 'red', 'room': 'kitchen'})",
                "",
                "print('graph nodes:', len(graph.nodes()))",
                "print('graph edges:', [(edge.relation, edge.confidence) for edge in graph.edges()])",
                "print('knowledge facts created:', created)",
                "print('reasoning target:', result.target_id)",
                "print('reasoning evidence:', result.evidence_ids)",
                "print('requires verification:', result.requires_verification)",
                "assert result.target_id is not None",
                "memory.close()"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "## 10. 运行时与离线评测要分开理解",
                "",
                "运行时系统只需要 RGB：",
                "",
                "```text",
                "RGB -> LingBot depth/c2w -> stable submap -> spatial memory",
                "RGB -> VLM -> semantic objects/triples -> graph/knowledge memory",
                "graph + knowledge memory -> native agent reasoning",
                "```",
                "",
                "GT depth、GT c2w 与 LiDAR 的作用是离线回答研究问题，例如：LingBot 的深度尺度是否漂移？局部子地图是否稳定？它们不应成为 agent 运行时必须连接的传感器。",
                "",
                "当前研究事实是：Stage1/VGGT 在短窗口内的几何 overlap 可以稳定，但对 Isaac GT 的绝对尺度和形状仍有明显误差。因此 runtime memory 会保留 confidence、overlap residual 和 provenance；agent 应在低置信度或证据冲突时选择重新观察/验证，而不是把每个几何关系当作绝对真相。"
            ]
    CameraIntrinsics,
    ObjectObservation,
)
import numpy as np

frame = FrameObservation(
    frame_id="demo_frame_1",
    timestamp=1.0,
    rgb=np.zeros((64, 96, 3), dtype=np.uint8),
    depth=np.full((64, 96), 2.0, dtype=np.float32),
    camera_intrinsics=CameraIntrinsics(80.0, 80.0, 48.0, 32.0, 96, 64),
    camera_pose=Pose3D(position=(0.0, 0.0, 0.0), yaw=0.0),
    robot_pose=Pose3D(position=(0.0, 0.0, 0.0), yaw=0.0),
)

obj = ObjectObservation(
    observation_id="obs_1",
    category="cup",
    attributes={"color": "red", "material": "ceramic"},
    bbox_2d=(10, 20, 40, 60),
    center_3d=(1.5, 0.7, 2.0),
    dimensions_3d=(0.12, 0.16, 0.12),
    confidence=0.96,
    timestamp=1.0,
    frame_id=frame.frame_id,
    embedding=np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32),
)

print("frame_id:", frame.frame_id)
print("category:", obj.category)
print("center_3d:", obj.center_3d)
print("bbox_3d:", obj.bbox_3d)

## 2. 看一看真正的代码结构

项目入口非常清晰，代码结构类似下面这样：

- `common/types.py`：统一的数据协议
- `mapping/`：生成点云、深度和 keyframe
- `perception/`：检测对象和语义信号
- `scene_graph/`：维护节点、边与几何关系
- `memory/`：存储 episodic / semantic / spatial memory
- `planning/`：基于 graph + memory 产生动作
- `execution/`：安全地执行导航动作
- `orchestration/pipeline.py`：把所有模块串起来

下面我们直接测试其中的关键模块：scene graph 和 memory。

In [ ]:
# 这一节：测试 SceneGraph 能否创建 node 和 edge
from agentic_memory_nav.scene_graph.graph import SceneGraph
from agentic_memory_nav.common.types import SceneNode, SceneEdge, NodeType
from agentic_memory_nav.common.types import new_id

# 构造两个节点：room 和 object
room = SceneNode(
    node_id=new_id("node"),
    node_type=NodeType.ROOM,
    label="kitchen",
    attributes={"kind": "room"},
    position_3d=(2.5, 0.0, 2.5),
    bbox_3d=(0.0, 0.0, 0.0, 5.0, 3.0, 5.0),
    uncertainty=0.0,
    first_seen=0.0,
    last_seen=0.0,
    confidence=0.99,
    source_frame="frame_0000",
    observation_count=1,
    observation_ids=["obs_room_1"],
)

cup = SceneNode(
    node_id=new_id("node"),
    node_type=NodeType.OBJECT,
    label="cup",
    attributes={"color": "red"},
    position_3d=(2.0, 0.85, 2.2),
    bbox_3d=(1.9, 0.6, 2.0, 2.1, 1.1, 2.4),
    uncertainty=0.05,
    first_seen=1.0,
    last_seen=1.0,
    confidence=0.96,
    source_frame="frame_0001",
    observation_count=1,
    observation_ids=["obs_cup_1"],
)

graph = SceneGraph()
graph.upsert_node(room)
graph.upsert_node(cup)

graph.upsert_edge(
    SceneEdge(
        edge_id=new_id("edge"),
        source_id=cup.node_id,
        target_id=room.node_id,
        relation="inside",
        confidence=0.95,
        first_seen=1.0,
        last_seen=1.0,
        source_frame="frame_0001",
        uncertainty=0.05,
        observation_count=1,
        position_3d=(2.0, 0.85, 2.2),
        provenance=["frame_0001", "manual_test"],
    )
)

print("nodes:", len(graph.nodes()))
print("edges:", len(graph.edges()))
print("first edge relation:", graph.edges()[0].relation)
assert len(graph.nodes()) == 2
assert len(graph.edges()) == 1

In [ ]:
# 这一节：测试 memory 能否存储、检索与更新
from pathlib import Path
from agentic_memory_nav.memory.sqlite_store import SQLiteMemory
from agentic_memory_nav.common.types import MemoryItem, MemoryType, new_id

memory_path = Path("/tmp/agentic_memory_nav_demo_memory.sqlite3")
if memory_path.exists():
    memory_path.unlink()

memory = SQLiteMemory(memory_path)

item = MemoryItem(
    memory_id=new_id("mem"),
    memory_type=MemoryType.SEMANTIC,
    content="red cup is inside the kitchen",
    structured_payload={"entity_id": "cup_1", "category": "cup", "color": "red"},
    timestamp=1.0,
    location=(2.0, 0.85, 2.2),
    confidence=0.95,
    provenance=["obs_cup_1"],
    embedding=[0.1, 0.9, 0.2],
)

memory.add_observation(item)
records = memory.retrieve_by_text("red cup kitchen")
print("retrieval count:", len(records))
print("first result:", records[0].content)
assert len(records) >= 1
assert "red cup" in records[0].content

In [ ]:
# 这一节：测试 graph updater 的几何关系推断逻辑
from agentic_memory_nav.scene_graph.updater import SceneGraphUpdater
from agentic_memory_nav.scene_graph.graph import SceneGraph
from agentic_memory_nav.common.types import ObjectObservation
from agentic_memory_nav.perception.mock_perception import MockPerception
from agentic_memory_nav.mapping.mock_mapper import MockMapper
import numpy as np

# Mock 场景：先做一帧，再做第二帧，类似 cup 被发现并持续更新
mapper = MockMapper(keyframe_interval=2, depth_m=2.0)
perception = MockPerception()
mapper.start()

graph = SceneGraph()
updater = SceneGraphUpdater(graph)

for idx in range(3):
    frame = __import__("agentic_memory_nav.common.types", fromlist=["FrameObservation"]).FrameObservation(
        str(idx), float(idx), np.zeros((32, 32, 3), dtype=np.uint8)
    )
    mapping = mapper.update(frame)
    obs = perception.detect(frame, mapping)
    updater.update(obs)

nodes = graph.find_nodes("cup", {"color": "red"})
print("cup nodes found:", len(nodes))
print("cup observation count:", nodes[0].observation_count)
print("edge relations:", [(e.source_id, e.target_id, e.relation) for e in graph.edges()[:5]])
assert len(nodes) >= 1
assert nodes[0].observation_count >= 2
assert any(edge.relation == "inside" for edge in graph.edges())

## 3. 我们现在真正验证了什么？

我们做了几个最核心的最小验证：

1. 项目能导入；
2. SceneGraph 能保存 node 和 edge；
3. Memory 能写入并检索；
4. `SceneGraphUpdater` 能把多帧对象观测合并，并推断 `inside` 等关系。

这说明当前实现确实具备“scene graph + memory + agentic planning”的基础骨架。它不是空架构。

但是它仍然不等于你想的那种“point-cloud-native scene graph”。

## 4. 你的想法和当前代码的差异

### 你想做的：point-cloud-centric architecture

你希望：

- node 表示 3D point cloud / region / instance
- edge 代表几何关系
- scene graph 可以直接承载长期几何记忆
- memory 可以从 graph 里直接检索具备 3D 语义的对象

这种设计更偏向：

- geometry-first
- spatial memory-first
- dense 3D relation semantics

### 当前项目做的：object-centric architecture

当前项目：

- node 表示对象、小区域、房间等抽象实体
- point cloud 更像是底层 map 层的一部分
- memory 存的是文字、结构化事实和空间索引
- scene graph 更偏“关系图”，而非“直接 3D 点云图”

这个差异并不是“谁错了”，而是工程层面的取舍：

- object-centric 更轻；
- point-cloud-centric 更丰富但更重；
- 当前仓库优先验证整体流程，默认是轻量、可测试和可复现的。

In [ ]:
# 这一节：快速总结“当前架构是否真的实现了你说的三层能力”
from agentic_memory_nav.scene_graph.graph import SceneGraph
from agentic_memory_nav.memory.sqlite_store import SQLiteMemory
from agentic_memory_nav.mapping.mock_mapper import MockMapper
from agentic_memory_nav.planning.rule_based_fallback import RuleBasedPlanner
from agentic_memory_nav.common.types import NavigationTask, TaskStatus, Pose3D
from pathlib import Path
import numpy as np

# 1. 是否构建了 scene graph
sg = SceneGraph()
print("scene graph exists:", hasattr(sg, "upsert_node"), hasattr(sg, "upsert_edge"))

# 2. 是否构建了 memory
mem = SQLiteMemory(Path("/tmp/agentic_memory_nav_verify.sqlite3"))
print("memory exists:", hasattr(mem, "add_observation"), hasattr(mem, "retrieve_by_text"))

# 3. 是否有 mapping
mapper = MockMapper()
print("mapping exists:", hasattr(mapper, "update"), hasattr(mapper, "get_global_pointcloud"))

# 4. 是否有 planner
planner = RuleBasedPlanner()
print("planner exists:", hasattr(planner, "plan"))

print("Conclusion: the repository implements a lightweight object-centric scene graph + memory + planner pipeline.")
assert hasattr(sg, "upsert_node") and hasattr(sg, "upsert_edge")
assert hasattr(mem, "add_observation") and hasattr(mem, "retrieve_by_text")
assert hasattr(mapper, "get_global_pointcloud")
assert hasattr(planner, "plan")

## 5. 你接下来最值得做的 5 个演进步骤

如果你想把当前实现升级成更接近你理想的 point-cloud-node architecture，建议按这个顺序推进：

1. 给 `SceneNode` 增加 `point_cloud_ref` 或 `cloud_id` 字段。
2. 让 `SceneGraph` 能承载 point-cloud summary，而不是只承载对象中心点。
3. 在 `MemoryItem` 中增加 geometry 相关字段，例如 `cloud_id`, `voxel_hash`, `region_id`。
4. 扩展 `SceneGraphUpdater`：用点云聚类、bbox、空间关系做更强的对象关联。
5. 把 planner 从“attribute-based”提升到“geometry + semantic retrieval”混合查询。

这一步非常重要：

- 它不要求你完全重写代码；
- 你只需要在当前结构上增加一个“几何引用层”，即可逐步演化成更先进的架构。

## 6. 最后一句话：当前代码是否实现了你的想法？

结论是：

- “场景图 + 记忆 + agentic planner” 实现了；
- “节点直接就是 3D 点云” 还没有完全实现；
- 但这两个架构是兼容的，当前代码更像是一个轻量版，而你的想法更像是增强版。

这说明你并不是“想法错了”，而是：

你正在从“可验证的轻量原型”逐步往“更强的几何化 scene graph”演化。

这正是一个正常研究架构演进路线。